In [2]:
!pip install Sastrawi
!pip install emoji
!pip install sentence-transformers

In [3]:
import pandas as pd
import numpy as np
import json
import re
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

import emoji

In [4]:
from google.colab import drive

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/Dataset_codingcamp_loker/data_untuk_modeling.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### Preprocessing Text

In [5]:
df = pd.read_csv(file_path)
df.columns.tolist()

['Unnamed: 0',
 'Judul',
 'Industri',
 'Tipe',
 'Kategori',
 'Kota',
 'Provinsi',
 'Negara',
 'Gaji_Min',
 'Gaji_Max',
 'Skills',
 'Pengalaman_Bulan',
 'Pendidikan',
 'Deskripsi']

In [9]:
# mengecek kolom kosong
df.isnull().sum()

,0
Unnamed: 0,0
Judul,0
Industri,0
Tipe,0
Kategori,0
Kota,0
Provinsi,0
Negara,0
Gaji_Min,0
Gaji_Max,0


In [ ]:
df.head(3)

,Unnamed: 0,Judul,Industri,Tipe,Kategori,Kota,Provinsi,Negara,Gaji_Min,Gaji_Max,Skills,Pengalaman_Bulan,Pendidikan,Deskripsi
0,0,IT Business Analyst,Information Technology and Services,FULL_TIME,System Analyst,Jakarta Timur,DKI Jakarta,ID,Tidak ada,Tidak ada,"SQL, Business Analysis, Relational Databases, ...",36.0,bachelor degree,Job description \n \n Evaluating and documenti...
1,1,IT System Analyst,Information Technology and Services,FULL_TIME,System Analyst,Surabaya,Jawa Timur,ID,7000000.0,9000000.0,"Requirements Analysis, SQL, Business Analysis,...",12.0,bachelor degree,General Qualification : \n \n Man (23 - 35 yea...
2,2,IT Business Analyst - Insurance exp,Information Technology and Services,CONTRACTOR,Business Analyst,Jakarta Selatan,DKI Jakarta,ID,13000000.0,14500000.0,"SDLC, Life Insurance, Teamwork, Business Proce...",36.0,bachelor degree,Skill And Experience : \n - Understand the Bas...


In [6]:
df['text_gabungan'] = (
    "judul: " + df['Judul'].fillna('') + " " +
    "industri: " + df['Industri'].fillna('') + " " +
    "kategori: " + df['Kategori'].fillna('') + " " +
    "pendidikan: " + df['Pendidikan'].fillna('') + " " +
    "tipe: " + df['Tipe'].fillna('') + " " +
    "lokasi: " + (df['Kota'].fillna('') + " " + df['Provinsi'].fillna('')) + " " +
    "skills: " + df['Skills'].fillna('') + " " +
    "deskripsi: " + df['Deskripsi'].fillna('')
)

print(df['text_gabungan'][0])

judul: IT Business Analyst industri: Information Technology and Services kategori: System Analyst pendidikan: bachelor degree tipe: FULL_TIME lokasi: Jakarta Timur DKI Jakarta skills: SQL, Business Analysis, Relational Databases, System Analysis, Analytical Skills, SDLC deskripsi: Job description 
 
 Evaluating and documenting business processes, anticipating requirements, uncovering areas for improvement, and developing and implementing solutions. 
 Conducting meetings and presentations to share ideas and findings. 
 Performing requirements analysis. 
 Working closely with clients, technical/programer, and project manager 
 Create Business Requirement documents and Functional Specification Document 
 
 
 Qualification 
 Minimum Bachelor’s Degree in Business ot  IT from reputable university 
 
 Minimum 2 (two) year experience as Business Analyst or Functional Analyst 
 Having good knowledge of Software Development Life Cycle  (SDLC) 
 Excellent documentation skills. 
 Excellent analyti

##### Cleaning

In [7]:

def basic_clean(text):
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)                 # HTML
    text = re.sub(r'[^a-z0-9\s:/_\-\+\.]', ' ', text) # keep tanda penting
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text_gabungan'] = df['text_gabungan'].apply(basic_clean)
print(df['text_gabungan'][0])

judul: it business analyst industri: information technology and services kategori: system analyst pendidikan: bachelor degree tipe: full_time lokasi: jakarta timur dki jakarta skills: sql business analysis relational databases system analysis analytical skills sdlc deskripsi: job description evaluating and documenting business processes anticipating requirements uncovering areas for improvement and developing and implementing solutions. conducting meetings and presentations to share ideas and findings. performing requirements analysis. working closely with clients technical/programer and project manager create business requirement documents and functional specification document qualification minimum bachelor s degree in business ot it from reputable university minimum 2 two year experience as business analyst or functional analyst having good knowledge of software development life cycle sdlc excellent documentation skills. excellent analytical judgment and problem solving skills high q

##### Case Folding

In [8]:
def case_folding(text):
    text = text.lower()
    return text

df["text_gabungan"] = df["text_gabungan"].apply(case_folding)
print(df["text_gabungan"][1])

judul: it system analyst industri: information technology and services kategori: system analyst pendidikan: bachelor degree tipe: full_time lokasi: surabaya jawa timur skills: requirements analysis sql business analysis sdlc stakeholder management cloud platform system teamwork system analysis deskripsi: general qualification : man 23 - 35 years old education min. s1 information technology/information system/computer system minimum 1-3 years of experience in it coordination business partnering or similar roles. strong understanding of it service management project management and technology roadmaps. strong communication collaboration and stakeholder management skills. fluent in bahasa indonesia and english written and spoken . will to work located in surabaya specific qualification : sql query/tuning basic programming language laravel basic understand client-server and networking concept basic it service management knowledge basic familiar with report development ssrs powerbi will be a

##### Normalization

In [ ]:
df_normalisasi = pd.read_csv("colloquial-indonesian-lexicon-normalisasi.csv")
df_normalisasi = df_normalisasi[["slang", "formal"]]
df_normalisasi.head()

,slang,formal
0,woww,wow
1,aminn,amin
2,met,selamat
3,netaas,menetas
4,keberpa,keberapa


In [ ]:
# normalization_dict = dict(zip(df_normalisasi['slang'], df_normalisasi['formal']))
# normalization_dict.pop('it')

# def normalization(text):
#     words = text.split()
#     words = [normalization_dict.get(word, word) for word in words]
#     return ' '.join(words)

# df['text_gabungan'] = df["text_gabungan"].apply(normalization)
# print(df['text_gabungan'][1])

judul: itu system analyst industri: information technology and services kategori: system analyst pendidikan: bachelor degree tipe: full_time lokasi: surabaya jawa timur skills: requirements analysis sql business analysis sdlc stakeholder management cloud platform system teamwork system analysis deskripsi: general qualification : man 23 - 35 years old education min. s1 information technology/information system/computer system minimum 1-3 years of experience ini itu coordination business partnering orang similar roles. strong understanding of itu service management project management and technology roadmaps. strong communication collaboration and stakeholder management skills. fluent ini bahasa indonesia and english written and spoken . will tapi work located ini surabaya specific qualification : sql query/tuning basic programming language laravel basic understand client-server and networking concept basic itu service management knowledge basic familiar with report development ssrs power

##### Stopword Removal

In [ ]:
# tidak relevan untuk kasus INDO ENGLISH, karena banyak kata yang sudah baku dan tidak ada slang yang perlu dinormalisasi dan Stopword

#### Membangun Modeling dengan Pseudo-CV
Input Text  
   ↓  
BERT Encoder  
   ↓  
Pooling (CLS / mean pooling)  
   ↓  
Dense (256) + ReLU  
   ↓  
Dense (128)  ← embedding akhir  
   ↓  
L2 Normalization  

#### Membangun Data Pseudo-CV

In [9]:
df['pseudo_cv'] = (
    "judul: " + df['Judul'].fillna('') + " " +
    "industri: " + df['Industri'].fillna('') + " " +
    "kategori: " + df['Kategori'].fillna('') + " " +
    "pendidikan: " + df['Pendidikan'].fillna('') + " " +
    "tipe: " + df['Tipe'].fillna('') + " " +
    "skills: " + df['Skills'].fillna('') + " "
)
print(df['pseudo_cv'][0])

judul: IT Business Analyst industri: Information Technology and Services kategori: System Analyst pendidikan: bachelor degree tipe: FULL_TIME skills: SQL, Business Analysis, Relational Databases, System Analysis, Analytical Skills, SDLC 


In [10]:
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.sentence_transformer import losses
from torch.utils.data import DataLoader

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
train_examples = [
    InputExample(texts=[df['pseudo_cv'], df['text_gabungan']])
    # for pseudo_cv, job_full in pairs
    for df['pseudo_cv'], df['text_gabungan'] in zip(df['pseudo_cv'], df['text_gabungan'])
]

train_loader = DataLoader(train_examples, batch_size=16, shuffle=True)
loss_fn = losses.MultipleNegativesRankingLoss(model)

model.fit(
    train_objectives=[(train_loader, loss_fn)],
    epochs=1
)

#### Inferens

In [ ]:
# cv_emb = model.encode(real_cv_text)
# job_embs = model.encode(job_list)

# scores = cosine_similarity([cv_emb], job_embs)